# Predictive Credit Risk Model with Snowflake ML

**Persona**: Dr. Priya Sharma, Lead Data Scientist at Simulated Asset Management

**Objective**: Build a predictive default model that goes beyond covenant-based monitoring. Uses XGBoost to predict probability of default from financial ratios, leverage metrics, ESG scores, and market regime state.

**Demo Flow** (7 steps):
1. **Feature Engineering with Feature Store** — Borrower-level features from financials + ESG + regime
2. **Training Data Construction** — Binary target from covenant deterioration patterns
3. **Model Training** — Walk-forward XGBoost with `SnowflakeXgboostCallback` autologging
4. **SHAP Explainability** — Waterfall (individual) + beeswarm (portfolio-level)
5. **Model Registry** — Versioned model with AUC/F1 metrics
6. **ML Pipeline Deployment** — DAG API for monthly scoring
7. **ML Observability** — Model Monitor for drift detection

**Source Data**: `DIM_CREDIT_BORROWER`, `FACT_CREDIT_BORROWER_FINANCIALS`, `FACT_CREDIT_COVENANT_TRACKING`, `FACT_ESG_SCORES`, `FACT_REGIME_PREDICTIONS`

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime

from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark.window import Window

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except:
    from snowflake.snowpark import Session
    session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "sfseeurope-mstellwall-aws-us-west3")).create()

DATABASE = "SAM_DEMO"
ML_SCHEMA = "ML"
CURATED = "CURATED"
MARKET_DATA = "MARKET_DATA"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {ML_SCHEMA}").collect()
print(f"Connected: {session.get_current_account()} | {DATABASE}.{ML_SCHEMA}")

Connected: "sfseeurope-mstellwall-aws-us-west3" | SAM_DEMO.ML


## Step 1: Feature Engineering with Feature Store

We build borrower-level credit features combining:
- **Financial ratios**: Leverage, coverage, margins from `FACT_CREDIT_BORROWER_FINANCIALS`
- **Covenant health**: Headroom and breach history from `FACT_CREDIT_COVENANT_TRACKING`
- **ESG risk**: Overall ESG score from `FACT_ESG_SCORES` (linked via issuer)
- **Market regime**: Current regime from `FACT_REGIME_PREDICTIONS` (cross-scenario dependency)

In [2]:
bf = (session.table(f"{DATABASE}.{CURATED}.FACT_CREDIT_BORROWER_FINANCIALS")
    .select(
        F.col("BORROWERID")
        , F.col("REPORTDATE").alias("QUARTER_DATE")
        , F.col("TOTALLEVERAGE"), F.col("NETLEVERAGE")
        , F.col("INTERESTCOVERAGE"), F.col("FIXEDCHARGECOVERAGE")
        , F.col("EBITDA_MARGIN")
        , F.col("FREECASHFLOW_MM"), F.col("CASHONHAND_MM")
        , F.col("REVOLVERAVAILABILITY_MM")
        , F.col("REVENUE_MM"), F.col("EBITDA_MM")
        , F.col("NETDEBT_MM"), F.col("CAPEX_MM")
        , F.when(F.col("REVENUE_MM") > 0, F.col("FREECASHFLOW_MM") / F.col("REVENUE_MM")).otherwise(F.lit(0)).alias("FCF_MARGIN")
        , F.when(F.col("EBITDA_MM") > 0, F.col("CAPEX_MM") / F.col("EBITDA_MM")).otherwise(F.lit(0)).alias("CAPEX_INTENSITY")
    )
)

ch = (session.table(f"{DATABASE}.{CURATED}.FACT_CREDIT_COVENANT_TRACKING")
    .group_by("BORROWERID", "TESTDATE")
    .agg(
        F.avg("HEADROOM_PCT").alias("AVG_COVENANT_HEADROOM")
        , F.sum(F.when(F.col("BREACHFLAG"), F.lit(1)).otherwise(F.lit(0))).alias("BREACH_COUNT")
        , F.count("*").alias("COVENANT_COUNT")
        , F.min("HEADROOM_PCT").alias("MIN_COVENANT_HEADROOM")
    )
    .select(
        F.col("BORROWERID").alias("CH_BORROWERID")
        , F.col("TESTDATE").alias("CH_QUARTER_DATE")
        , "AVG_COVENANT_HEADROOM", "BREACH_COUNT", "COVENANT_COUNT", "MIN_COVENANT_HEADROOM"
    )
)

bi = (session.table(f"{DATABASE}.{CURATED}.DIM_CREDIT_BORROWER")
    .select(
        F.col("BorrowerID").alias("BI_BORROWERID")
        , F.col("Sector").alias("SECTOR")
        , F.col("CreditRating").alias("CREDITRATING")
    )
)

regime = (session.table(f"{DATABASE}.{ML_SCHEMA}.FACT_REGIME_PREDICTIONS")
    .select(
        F.col("DATE").alias("REGIME_DATE")
        , F.col("REGIME_LABEL")
    )
    .with_column("REGIME_QTR", F.call_builtin("DATE_TRUNC", F.lit("quarter"), F.col("REGIME_DATE")))
    .with_column("RN", F.row_number().over(Window.partition_by("REGIME_QTR").order_by(F.col("REGIME_DATE").desc())))
    .filter(F.col("RN") == 1)
    .select("REGIME_QTR", "REGIME_LABEL")
)

esg = (session.table(f"{DATABASE}.{CURATED}.FACT_ESG_SCORES")
    .pivot("SCORE_TYPE", ["Environmental", "Social", "Governance", "Overall ESG"])
    .agg(F.avg("SCORE_VALUE"))
    .select(
        F.col("SECURITYID").alias("ESG_SID")
        , F.col("SCORE_DATE").alias("ESG_DATE")
        , F.col("'Environmental'").alias("E_SCORE")
        , F.col("'Social'").alias("S_SCORE")
        , F.col("'Governance'").alias("G_SCORE")
        , F.col("'Overall ESG'").alias("ESG_COMPOSITE")
    )
)

sector_esg = (esg
    .join(session.table(f"{DATABASE}.{CURATED}.DIM_SECURITY"), esg["ESG_SID"] == F.col("SECURITYID"), "inner")
    .with_column("ESG_QTR", F.call_builtin("DATE_TRUNC", F.lit("quarter"), F.col("ESG_DATE")))
    .group_by("ASSETCLASS", "ESG_QTR")
    .agg(
        F.avg("E_SCORE").alias("E_SCORE")
        , F.avg("S_SCORE").alias("S_SCORE")
        , F.avg("G_SCORE").alias("G_SCORE")
        , F.avg("ESG_COMPOSITE").alias("ESG_COMPOSITE")
    )
    .select(
        F.col("ESG_QTR")
        , F.col("E_SCORE").cast(T.DecimalType(38, 10))
        , F.col("S_SCORE").cast(T.DecimalType(38, 10))
        , F.col("G_SCORE").cast(T.DecimalType(38, 10))
        , F.col("ESG_COMPOSITE").cast(T.DecimalType(38, 10))
    )
)

print(f"Borrower financials: {bf.count()} rows")
print(f"Covenant health: {ch.count()} rows")
print(f"Borrower info: {bi.count()} rows")
print(f"Regime quarters: {regime.count()} rows")
print(f"Sector ESG quarters: {sector_esg.count()} rows")

Borrower financials: 180 rows
Covenant health: 120 rows
Borrower info: 15 rows
Regime quarters: 0 rows
Sector ESG quarters: 20 rows


### Join Sources and Build Feature Matrix

Left-join borrower financials with covenant health metrics, borrower sector/rating info, and the latest market regime per quarter. Coalesce nulls to sensible defaults (e.g., 0.5 headroom, 0 breaches, "UNKNOWN" regime).

In [ ]:
DECIMAL_TYPE = T.DecimalType(38, 10)

float_cols = [
    "TOTALLEVERAGE", "NETLEVERAGE", "INTERESTCOVERAGE", "FIXEDCHARGECOVERAGE",
    "EBITDA_MARGIN", "FCF_MARGIN", "CAPEX_INTENSITY",
    "FREECASHFLOW_MM", "CASHONHAND_MM", "REVOLVERAVAILABILITY_MM",
    "AVG_COVENANT_HEADROOM", "MIN_COVENANT_HEADROOM",
    "E_SCORE", "S_SCORE", "G_SCORE", "ESG_COMPOSITE",
]

feature_df = (bf
    .join(ch
        , (bf["BORROWERID"] == ch["CH_BORROWERID"]) & (bf["QUARTER_DATE"] == ch["CH_QUARTER_DATE"])
        , "left"
    )
    .join(bi, bf["BORROWERID"] == bi["BI_BORROWERID"], "left")
    .join(regime
        , F.call_builtin("DATE_TRUNC", F.lit("quarter"), bf["QUARTER_DATE"]) == regime["REGIME_QTR"]
        , "left"
    )
    .join(sector_esg
        , F.call_builtin("DATE_TRUNC", F.lit("quarter"), bf["QUARTER_DATE"]) == sector_esg["ESG_QTR"]
        , "left"
    )
    .select(
        bf["BORROWERID"], bf["QUARTER_DATE"]
        , bf["TOTALLEVERAGE"], bf["NETLEVERAGE"]
        , bf["INTERESTCOVERAGE"], bf["FIXEDCHARGECOVERAGE"]
        , bf["EBITDA_MARGIN"], bf["FCF_MARGIN"], bf["CAPEX_INTENSITY"]
        , bf["FREECASHFLOW_MM"], bf["CASHONHAND_MM"], bf["REVOLVERAVAILABILITY_MM"]
        , F.coalesce(ch["AVG_COVENANT_HEADROOM"], F.lit(0.5)).alias("AVG_COVENANT_HEADROOM")
        , F.coalesce(ch["BREACH_COUNT"], F.lit(0)).alias("BREACH_COUNT")
        , F.coalesce(ch["MIN_COVENANT_HEADROOM"], F.lit(0.5)).alias("MIN_COVENANT_HEADROOM")
        , F.col("SECTOR"), F.col("CREDITRATING")
        , F.coalesce(regime["REGIME_LABEL"], F.lit("UNKNOWN")).alias("MARKET_REGIME")
        , F.coalesce(sector_esg["E_SCORE"], F.lit(0.5)).alias("E_SCORE")
        , F.coalesce(sector_esg["S_SCORE"], F.lit(0.5)).alias("S_SCORE")
        , F.coalesce(sector_esg["G_SCORE"], F.lit(0.5)).alias("G_SCORE")
        , F.coalesce(sector_esg["ESG_COMPOSITE"], F.lit(0.5)).alias("ESG_COMPOSITE")
    )
)

for col_name in float_cols:
    feature_df = feature_df.with_column(col_name, F.col(col_name).cast(DECIMAL_TYPE))

feature_pd = feature_df.to_pandas()
print(f"Credit risk features: {len(feature_pd)} rows, {feature_pd.shape[1]} columns")
print(f"Borrowers: {feature_pd['BORROWERID'].nunique()}, Quarters: {feature_pd['QUARTER_DATE'].nunique()}")
feature_pd.describe()

#### Data Quality Gate: Join Coverage and Default-Fill Rates

Credit features come from 5 source tables joined on borrower ID and quarter. Missing ESG scores, regime labels, or covenant data are filled with defaults via COALESCE (0.5 for ESG, 0 for breach count, "UNKNOWN" for regime). If many rows use these defaults, the model learns from synthetic placeholder values rather than real signals.

**What to look for:**
- **Per-source coverage** — What % of borrower-quarters have actual (non-default) values for each source? Covenant and ESG data may be sparser than financial ratios.
- **Default-fill rates** — Features where >50% of values are the COALESCE default (0.5, 0, "UNKNOWN") contribute noise rather than signal. The model may learn to predict based on "missing data patterns" rather than actual credit risk.
- **Borrower count over time** — Should be roughly stable. A sudden drop suggests data lag; a sudden spike suggests a schema change or new data load.

**Decision impact:** Features with >50% default-fill rates should be flagged — consider dropping them or imputing more thoughtfully. If covenant coverage is very low, the model is essentially learning from financial ratios and ESG only, and the covenant features add noise.

In [ ]:
default_checks = {
    "AVG_COVENANT_HEADROOM": 0.5,
    "MIN_COVENANT_HEADROOM": 0.5,
    "BREACH_COUNT": 0,
    "E_SCORE": 0.5,
    "S_SCORE": 0.5,
    "G_SCORE": 0.5,
    "ESG_COMPOSITE": 0.5,
}

fill_aggs = [F.count("*").alias("TOTAL")]
for col, default_val in default_checks.items():
    fill_aggs.append(
        F.sum(F.when(F.col(col) == F.lit(default_val), 1).otherwise(0)).alias(f"{col}_DEFAULT")
    )

fill_stats = feature_df.select(*fill_aggs).to_pandas().iloc[0]
total = int(fill_stats["TOTAL"])

default_fill_pcts = {col: float(fill_stats[f"{col}_DEFAULT"]) / total * 100 for col in default_checks}

borrower_timeline = (feature_df
    .group_by("QUARTER_DATE")
    .agg(F.count_distinct("BORROWERID").alias("N_BORROWERS"))
    .sort("QUARTER_DATE")
).to_pandas()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Default-Fill Rate by Feature (%)", "Borrowers per Quarter"))

fill_colors = ["#e74c3c" if v > 50 else "#f39c12" if v > 30 else "#2ecc71" for v in default_fill_pcts.values()]
fig.add_trace(go.Bar(y=list(default_fill_pcts.keys()), x=list(default_fill_pcts.values()),
    orientation="h", marker_color=fill_colors, showlegend=False), row=1, col=1)
fig.add_vline(x=50, line_dash="dash", line_color="red", opacity=0.5, row=1, col=1)
fig.update_xaxes(title_text="% of rows using COALESCE default", row=1, col=1)

fig.add_trace(go.Scatter(x=borrower_timeline["QUARTER_DATE"], y=borrower_timeline["N_BORROWERS"],
    mode="lines+markers", marker=dict(size=4), showlegend=False), row=1, col=2)
fig.update_yaxes(title_text="Distinct Borrowers", row=1, col=2)
fig.update_xaxes(tickangle=45, tickfont_size=8, row=1, col=2)

fig.update_layout(height=380, template="plotly_white")
fig.show()

print(f"{'Feature':<30} {'Default-Fill %':>15} {'Status':>10}")
print("-" * 60)
for col, pct in default_fill_pcts.items():
    status = "FAIL" if pct > 50 else "REVIEW" if pct > 30 else "PASS"
    print(f"{col:<30} {pct:>14.1f}% {status:>10}")

### Feature Store Initialisation

Connect to the Feature Store schema. Uses `CREATE_IF_NOT_EXIST` so re-runs are idempotent.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=ML_SCHEMA,
    default_warehouse="SAM_DEMO_EXECUTION_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

print(f"Feature Store: {DATABASE}.{ML_SCHEMA}")

### Register Entity

In [ ]:
borrower_entity = Entity(name="BORROWER", join_keys=["BORROWERID"], desc="Private credit borrower")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    try:
        fs.register_entity(borrower_entity)
        print(f"Entity registered: {borrower_entity.name}")
    except Exception:
        borrower_entity = fs.get_entity("BORROWER")
        print(f"Entity already exists, retrieved: {borrower_entity.name}")

### Register FeatureView

Cast `QUARTER_DATE` to `TIMESTAMP` (required by Feature Store) and register the credit risk features as a versioned FeatureView backed by a managed Dynamic Table.

In [ ]:
credit_fv_df = feature_df.with_column("QUARTER_DATE", F.col("QUARTER_DATE").cast(T.TimestampType()))

credit_fv = FeatureView(
    name="CREDIT_RISK_FEATURES"
    , entities=[borrower_entity]
    , feature_df=credit_fv_df
    , timestamp_col="QUARTER_DATE"
    , refresh_freq="1 day"
    , warehouse="SAM_DEMO_EXECUTION_WH"
    , desc="Borrower-level credit risk features: financials, covenant health, regime state"
)

registered_credit_fv = fs.register_feature_view(credit_fv, version="V01", overwrite=True)
registered_credit_fv.attach_feature_desc({
    "TOTALLEVERAGE": "Total debt / EBITDA ratio",
    "NETLEVERAGE": "Net debt / EBITDA ratio",
    "INTERESTCOVERAGE": "EBITDA / interest expense",
    "FIXEDCHARGECOVERAGE": "EBITDA / (interest + rent + capex)",
    "EBITDA_MARGIN": "EBITDA / Revenue",
    "FCF_MARGIN": "Free cash flow / Revenue",
    "CAPEX_INTENSITY": "Capex / EBITDA",
    "FREECASHFLOW_MM": "Free cash flow in millions",
    "CASHONHAND_MM": "Cash on hand in millions",
    "REVOLVERAVAILABILITY_MM": "Revolver availability in millions",
    "AVG_COVENANT_HEADROOM": "Average covenant headroom percentage",
    "BREACH_COUNT": "Number of covenant breaches",
    "MIN_COVENANT_HEADROOM": "Minimum covenant headroom percentage",
    "E_SCORE": "Environmental ESG score (sector avg proxy)",
    "S_SCORE": "Social ESG score (sector avg proxy)",
    "G_SCORE": "Governance ESG score (sector avg proxy)",
    "ESG_COMPOSITE": "Overall ESG composite score (sector avg proxy)",
    "SECTOR": "Borrower industry sector",
    "CREDITRATING": "Current credit rating",
    "MARKET_REGIME": "Market regime state from regime model",
})
print(f"FeatureView registered: CREDIT_RISK_FEATURES/V01")

## Step 2: Training Data Construction

We construct a binary default target from covenant deterioration patterns:
- **Default (1)**: Borrower had a covenant breach OR min headroom < 5% in a quarter
- **No default (0)**: All covenants healthy

In [ ]:
spine_df = session.table(f"{DATABASE}.{CURATED}.FACT_CREDIT_BORROWER_FINANCIALS").select(
    F.col("BORROWERID"),
    F.col("REPORTDATE").alias("QUARTER_DATE"),
)

dataset = fs.generate_dataset(
    name=f"{DATABASE}.{ML_SCHEMA}.CREDIT_RISK_TRAINING_DS",
    spine_df=spine_df,
    features=[registered_credit_fv],
    spine_timestamp_col="QUARTER_DATE",
    version="V01",
    desc="Credit risk training dataset from Feature Store"
)

training_data = dataset.read.to_pandas()

training_data["DEFAULT_FLAG"] = (
    (training_data["BREACH_COUNT"] > 0) | 
    (training_data["MIN_COVENANT_HEADROOM"] < 0.05)
).astype(int)

numeric_cols = ["TOTALLEVERAGE", "NETLEVERAGE", "INTERESTCOVERAGE", "FIXEDCHARGECOVERAGE",
                "EBITDA_MARGIN", "FCF_MARGIN", "CAPEX_INTENSITY", "FREECASHFLOW_MM",
                "CASHONHAND_MM", "REVOLVERAVAILABILITY_MM", "AVG_COVENANT_HEADROOM",
                "BREACH_COUNT", "MIN_COVENANT_HEADROOM",
                "E_SCORE", "S_SCORE", "G_SCORE", "ESG_COMPOSITE"]

cat_cols = ["SECTOR", "CREDITRATING", "MARKET_REGIME"]

from sklearn.preprocessing import LabelEncoder
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    training_data[col] = le.fit_transform(training_data[col].fillna("UNKNOWN"))
    label_encoders[col] = le

X = training_data[numeric_cols + cat_cols].fillna(0)
y = training_data["DEFAULT_FLAG"]

print(f"Training samples: {len(X)}")
print(f"Default rate: {y.mean():.1%}")
print(f"Defaults: {y.sum()}, Non-defaults: {(1-y).sum()}")

#### Target Quality Gate: Class Balance and Default Distribution

The default flag is constructed from covenant deterioration (breach OR headroom < 5%). If the default rate is extremely low (<2%), the model faces severe class imbalance and may predict "no default" for everyone. If extremely high (>40%), the definition is too loose and the model cannot distinguish truly distressed borrowers.

**What to look for:**
- **Default rate** — Should be 5-30% for a well-calibrated binary target. Lower rates require class balancing (handled via `scale_pos_weight`); higher rates suggest the target captures covenant stress rather than actual default risk.
- **Temporal distribution** — Defaults should be spread across multiple quarters, not concentrated in one period (e.g., all defaults in Q2 2020 would make the model a "COVID detector" rather than a credit model).
- **Borrower concentration** — If 1-2 chronic defaulters dominate the positive class, the model learns borrower-specific patterns rather than generalisable credit risk signals.

**Decision impact:** If default rate < 2%, consider expanding the definition (e.g., headroom < 10%) or using downsampling. If > 40%, tighten the definition. If defaults are concentrated in < 3 quarters, add time-based stratification to cross-validation. If dominated by < 5 borrowers, consider borrower-level CV splits.

In [ ]:
default_rate = y.mean() * 100

fig = make_subplots(rows=1, cols=3,
    subplot_titles=(f"Class Balance (default rate={default_rate:.1f}%)",
                    "Default Rate by Quarter",
                    f"Top 10 Defaulters ({borrower_defaults.nunique()} unique borrowers)"),
    specs=[[{"type": "pie"}, {"type": "xy"}, {"type": "xy"}]])

fig.add_trace(go.Pie(labels=[f"Default ({int(y.sum())})", f"No Default ({int((1-y).sum())})"],
    values=[y.sum(), (1-y).sum()], marker_colors=["#e74c3c", "#2ecc71"],
    textinfo="percent+label"), row=1, col=1)

quarterly_defaults = training_data.groupby("QUARTER_DATE")["DEFAULT_FLAG"].agg(["sum", "count"])
quarterly_defaults["RATE"] = quarterly_defaults["sum"] / quarterly_defaults["count"] * 100
fig.add_trace(go.Bar(x=quarterly_defaults.index.astype(str), y=quarterly_defaults["RATE"],
    marker_color="#e74c3c", opacity=0.7, showlegend=False), row=1, col=2)
fig.update_xaxes(tickangle=45, tickfont_size=7, row=1, col=2)
fig.update_yaxes(title_text="Default Rate (%)", row=1, col=2)

borrower_defaults = training_data[training_data["DEFAULT_FLAG"] == 1].groupby("BORROWERID").size().sort_values(ascending=False)
top_borrowers = borrower_defaults.head(10)
fig.add_trace(go.Bar(y=[f"Borrower {b}" for b in top_borrowers.index], x=top_borrowers.values,
    orientation="h", marker_color="#f39c12", showlegend=False), row=1, col=3)
fig.update_xaxes(title_text="# Default Quarters", row=1, col=3)

fig.update_layout(height=380, template="plotly_white")
fig.show()

n_default_quarters = (quarterly_defaults["sum"] > 0).sum()
n_unique_defaulters = borrower_defaults.nunique()
top5_concentration = borrower_defaults.head(5).sum() / y.sum() * 100 if y.sum() > 0 else 0

print(f"{'Metric':<40} {'Value':>12} {'Status':>10}")
print("-" * 65)
print(f"{'Default rate':<40} {default_rate:>11.1f}% {'PASS' if 5 <= default_rate <= 30 else 'REVIEW':>10}")
print(f"{'Quarters with defaults':<40} {n_default_quarters:>12} {'PASS' if n_default_quarters > 3 else 'REVIEW':>10}")
print(f"{'Unique defaulting borrowers':<40} {n_unique_defaulters:>12} {'PASS' if n_unique_defaulters > 5 else 'REVIEW':>10}")
print(f"{'Top-5 borrower concentration':<40} {top5_concentration:>11.1f}% {'REVIEW' if top5_concentration > 50 else 'PASS':>10}")

#### Feature Quality Gate: Correlation and Multicollinearity

Credit features are often highly correlated — total leverage vs net leverage, EBITDA margin vs FCF margin, various ESG sub-scores. While XGBoost is less affected by multicollinearity than linear models, high correlation makes SHAP explanations unreliable: SHAP may arbitrarily attribute importance to one of a correlated pair, making explanations inconsistent across different data subsets.

**What to look for:**
- **Pairwise Spearman correlation** — Pairs with |rho| > 0.8 should be flagged. For credit models, TOTALLEVERAGE/NETLEVERAGE and E_SCORE/ESG_COMPOSITE are likely candidates.
- **Condition number** — Values > 30 indicate multicollinearity that affects SHAP stability. This matters for regulatory explainability — auditors expect consistent feature attributions.

**Decision impact:** Pairs with |rho| > 0.8 should be reviewed — consider dropping one or combining them. This is especially important for credit models where SHAP explanations are used for regulatory reporting (SR 11-7, ECAI guidelines). Unstable SHAP rankings across subsets undermine trust in model explanations.

In [ ]:
all_features = numeric_cols + cat_cols
corr = X.corr(method="spearman")

text_vals = [[f"{corr.values[r, c]:.2f}" if abs(corr.values[r, c]) > 0.5 else ""
             for c in range(len(all_features))] for r in range(len(all_features))]

fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=all_features, y=all_features,
    colorscale="RdBu_r", zmin=-1, zmax=1,
    text=text_vals, texttemplate="%{text}", textfont_size=7,
    colorbar=dict(title="ρ")))
fig.update_layout(height=600, width=700, template="plotly_white",
    title_text="Credit Feature Spearman Correlation", title_font_size=12,
    xaxis=dict(tickangle=45, tickfont_size=7), yaxis=dict(tickfont_size=7))
fig.show()

cond_number = np.linalg.cond(X.dropna().values)
high_pairs = []
for i in range(len(all_features)):
    for j in range(i + 1, len(all_features)):
        if abs(corr.values[i, j]) > 0.8:
            high_pairs.append((all_features[i], all_features[j], corr.values[i, j]))

print(f"\nCondition number: {cond_number:.1f} {'PASS' if cond_number < 30 else 'REVIEW' if cond_number < 100 else 'FAIL'}")
print(f"\nHighly correlated pairs (|rho| > 0.8):")
if high_pairs:
    for a, b, rho in high_pairs:
        print(f"  {a} <-> {b}: rho = {rho:+.3f}  REVIEW")
else:
    print("  None found  PASS")

## Step 3: Model Training — XGBoost with Experiment Tracking

Walk-forward cross-validation with `SnowflakeXgboostCallback` for automatic logging of training metrics at each boosting round.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from snowflake.ml.experiment import ExperimentTracking, SnowflakeXgboostCallback

experiment = ExperimentTracking(
    session=session,
    experiment_name="credit_risk",
    database_name=DATABASE,
    schema_name=ML_SCHEMA
)
experiment.set_experiment("credit_risk")

xgb_callback = SnowflakeXgboostCallback(experiment)

params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "max_depth": 6,
    "learning_rate": 0.05,
    "n_estimators": 200,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "scale_pos_weight": (1 - y.mean()) / max(y.mean(), 0.01),
    "random_state": 42,
}

for k, v in params.items():
    experiment.log_param(k, v)

tscv = TimeSeriesSplit(n_splits=3)
auc_scores = []
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(**params, callbacks=[xgb_callback])
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)
    
    fold_auc = roc_auc_score(y_val, y_pred_proba)
    fold_f1 = f1_score(y_val, y_pred, zero_division=0)
    auc_scores.append(fold_auc)
    f1_scores.append(fold_f1)
    print(f"Fold {fold+1}: AUC={fold_auc:.4f}, F1={fold_f1:.4f}")

mean_auc = np.mean(auc_scores)
mean_f1 = np.mean(f1_scores)
experiment.log_metric("mean_auc", mean_auc)
experiment.log_metric("mean_f1", mean_f1)

print(f"\nMean AUC: {mean_auc:.4f}, Mean F1: {mean_f1:.4f}")

final_model = xgb.XGBClassifier(**params)
final_model.fit(X, y, verbose=False)
print("Final model trained on full dataset")

#### Model Quality Gate: Performance, Calibration, and Fold Consistency

AUC and F1 alone do not tell the full story for credit risk models. We need calibration (does a 20% PD actually default ~20% of the time?) and per-fold stability (is performance consistent across time periods?).

**What to look for:**
- **ROC curve** — AUC > 0.7 is acceptable for credit models; > 0.8 is good. The curve should bow well above the diagonal.
- **Calibration** — Plot predicted PD vs actual default rate in decile bins. The points should lie near the 45-degree line. Systematic over- or under-prediction means the PD values are unreliable for capital calculations.
- **Per-fold consistency** — AUC and F1 should be similar across CV folds. Large variation (AUC std > 0.1) means the model's performance depends on which time period it is evaluated on.
- **Precision-recall curve** — Important for imbalanced classes. High recall (catching most defaults) may be more important than high precision for risk management.

**Decision impact:** If AUC < 0.7, the model has weak discrimination — revisit features or target definition. If calibration is poor (slope far from 1.0), consider Platt scaling or isotonic regression. If per-fold AUC varies by > 0.15, the model is unstable across time and may need regime-conditional training.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix
from sklearn.calibration import calibration_curve

y_pred_prob_full = final_model.predict_proba(X)[:, 1]
y_pred_full = final_model.predict(X)

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "ROC Curve", "Calibration Curve",
    f"Per-Fold Metrics (AUC std={np.std(auc_scores):.3f})", "Confusion Matrix"],
    specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "xy"}, {"type": "heatmap"}]])

fpr, tpr, _ = roc_curve(y, y_pred_prob_full)
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", line=dict(width=2),
    name=f"AUC = {mean_auc:.3f}"), row=1, col=1)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
    line=dict(dash="dash", color="gray"), showlegend=False), row=1, col=1)
fig.update_xaxes(title_text="False Positive Rate", row=1, col=1)
fig.update_yaxes(title_text="True Positive Rate", row=1, col=1)

prob_true, prob_pred = calibration_curve(y, y_pred_prob_full, n_bins=10, strategy="quantile")
fig.add_trace(go.Scatter(x=prob_pred, y=prob_true, mode="lines+markers",
    line=dict(width=2), name="Model"), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
    line=dict(dash="dash", color="gray"), name="Perfect", showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="Mean Predicted PD", row=1, col=2)
fig.update_yaxes(title_text="Actual Default Rate", row=1, col=2)

fold_labels = [f"Fold {i+1}" for i in range(len(auc_scores))]
fig.add_trace(go.Bar(x=fold_labels, y=auc_scores, name="AUC",
    marker_color="#3498db"), row=2, col=1)
fig.add_trace(go.Bar(x=fold_labels, y=f1_scores, name="F1",
    marker_color="#e67e22"), row=2, col=1)
fig.add_hline(y=mean_auc, line_dash="dash", line_color="#3498db", opacity=0.5, row=2, col=1)
fig.add_hline(y=mean_f1, line_dash="dash", line_color="#e67e22", opacity=0.5, row=2, col=1)
fig.update_layout(barmode="group")

cm = confusion_matrix(y, y_pred_full)
cm_text = [[str(cm[r, c]) for c in range(cm.shape[1])] for r in range(cm.shape[0])]
fig.add_trace(go.Heatmap(z=cm,
    x=["Predicted No Default", "Predicted Default"],
    y=["Actual No Default", "Actual Default"],
    colorscale="Blues", text=cm_text, texttemplate="%{text}", textfont_size=14,
    showscale=False), row=2, col=2)

fig.update_layout(height=700, template="plotly_white",
    title_text="Credit Risk Model Performance Diagnostics", title_font_size=14)
fig.show()

auc_std = np.std(auc_scores)
calib_slope = np.polyfit(prob_pred, prob_true, 1)[0] if len(prob_pred) > 1 else 0

print(f"{'Metric':<35} {'Value':>12} {'Status':>10}")
print("-" * 60)
print(f"{'Mean AUC':<35} {mean_auc:>12.4f} {'PASS' if mean_auc > 0.7 else 'REVIEW':>10}")
print(f"{'Mean F1':<35} {mean_f1:>12.4f} {'PASS' if mean_f1 > 0.3 else 'REVIEW':>10}")
print(f"{'AUC std across folds':<35} {auc_std:>12.4f} {'PASS' if auc_std < 0.1 else 'REVIEW':>10}")
print(f"{'Calibration slope':<35} {calib_slope:>12.2f} {'PASS' if 0.5 < calib_slope < 1.5 else 'REVIEW':>10}")

## Step 4: SHAP Explainability

SHAP values show which features drive default predictions for individual borrowers (waterfall) and across the portfolio (beeswarm).

In [ ]:
import shap

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)

all_feature_names = numeric_cols + cat_cols

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

plt.sca(axes[0])
shap.summary_plot(shap_values, X, feature_names=all_feature_names, show=False, plot_type="dot")
axes[0].set_title("SHAP Beeswarm — Portfolio-Level Feature Importance")

plt.sca(axes[1])
high_risk_idx = y.idxmax() if y.sum() > 0 else 0
shap.plots.waterfall(shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=explainer.expected_value,
    data=X.iloc[high_risk_idx],
    feature_names=all_feature_names
), show=False)
axes[1].set_title("SHAP Waterfall — Highest Risk Borrower")

plt.tight_layout()
plt.show()

shap_importance = pd.DataFrame({
    "feature": all_feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

print("Top 10 Features by SHAP Importance:")
print(shap_importance.head(10).to_string(index=False))

#### Explainability Gate: SHAP Stability Across Time Periods

If SHAP feature rankings change significantly across CV folds (different time periods), the model's explanations are unstable — different periods produce different stories about what drives default risk. For credit models used in regulatory reporting, this is a critical concern.

**What to look for:**
- **Top-5 feature overlap** — The same features should appear in the top 5 across all folds. If leverage is #1 in fold 1 but #8 in fold 3, the explanation is period-dependent.
- **SHAP direction consistency** — For each feature, the sign of mean SHAP should be consistent across folds. If high leverage increases default risk in fold 1 but decreases it in fold 2, the model has learned contradictory patterns.
- **Feature importance ranking correlation** — Spearman correlation of SHAP importance rankings across folds. > 0.7 means rankings are consistent; < 0.5 means explanations are unreliable.

**Decision impact:** If top-5 overlap < 60% across folds, SHAP explanations should not be used for individual loan decisions — only portfolio-level analysis. If direction is inconsistent for a major feature, investigate whether the relationship is regime-dependent (e.g., leverage matters differently in RISK_OFF vs RISK_ON periods).

In [ ]:
from scipy import stats as sp_stats

fold_shap_rankings = []
fold_shap_signs = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    fold_model = xgb.XGBClassifier(**params)
    fold_model.fit(X.iloc[train_idx], y.iloc[train_idx], verbose=False)
    fold_explainer = shap.TreeExplainer(fold_model)
    fold_shap = fold_explainer.shap_values(X.iloc[val_idx])
    mean_abs = np.abs(fold_shap).mean(axis=0)
    mean_signed = fold_shap.mean(axis=0)
    fold_shap_rankings.append(pd.Series(mean_abs, index=all_feature_names).rank(ascending=False))
    fold_shap_signs.append(np.sign(mean_signed))

ranking_df = pd.DataFrame(fold_shap_rankings, index=[f"Fold {i+1}" for i in range(len(fold_shap_rankings))])

fold_labels_ov = [f"Fold {i+1}" for i in range(len(top5_per_fold))]
overlap_text = [[f"{overlap_matrix[r, c]:.0f}%" for c in range(overlap_matrix.shape[1])]
                for r in range(overlap_matrix.shape[0])]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Top-5 Feature Overlap Across Folds (%)",
                    "SHAP Direction Consistency (% folds agree on sign)"),
    specs=[[{"type": "heatmap"}, {"type": "xy"}]])

fig.add_trace(go.Heatmap(z=overlap_matrix, x=fold_labels_ov, y=fold_labels_ov,
    colorscale="YlGn", zmin=0, zmax=100,
    text=overlap_text, texttemplate="%{text}", textfont_size=12,
    colorbar=dict(x=0.45, len=0.8)), row=1, col=1)

sign_consistency = {}
for col in all_feature_names:
    signs = [s[all_feature_names.index(col)] for s in fold_shap_signs]
    agreement = max(sum(1 for s in signs if s > 0), sum(1 for s in signs if s <= 0)) / len(signs) * 100
    sign_consistency[col] = agreement

sorted_features = sorted(sign_consistency.items(), key=lambda x: -x[1])
feat_names = [f[0] for f in sorted_features[:15]]
feat_vals = [f[1] for f in sorted_features[:15]]
bar_colors = ["#2ecc71" if v == 100 else "#f39c12" if v >= 66 else "#e74c3c" for v in feat_vals]
fig.add_trace(go.Bar(y=feat_names, x=feat_vals, orientation="h",
    marker_color=bar_colors, showlegend=False), row=1, col=2)
fig.add_vline(x=66, line_dash="dash", line_color="orange", opacity=0.5, row=1, col=2)
fig.update_xaxes(title_text="% Agreement", row=1, col=2)

fig.update_layout(height=420, template="plotly_white")
fig.show()

rank_corrs = []
for i in range(len(ranking_df)):
    for j in range(i+1, len(ranking_df)):
        rho, _ = sp_stats.spearmanr(ranking_df.iloc[i], ranking_df.iloc[j])
        rank_corrs.append(rho)
avg_rank_corr = np.mean(rank_corrs)
avg_overlap = np.mean([overlap_matrix[i, j] for i in range(len(top5_per_fold)) for j in range(i+1, len(top5_per_fold))])

print(f"{'Metric':<40} {'Value':>12} {'Status':>10}")
print("-" * 65)
print(f"{'Avg top-5 overlap across folds':<40} {avg_overlap:>11.0f}% {'PASS' if avg_overlap > 60 else 'REVIEW':>10}")
print(f"{'Avg ranking correlation (Spearman)':<40} {avg_rank_corr:>12.3f} {'PASS' if avg_rank_corr > 0.7 else 'REVIEW':>10}")
consistent_features = sum(1 for v in sign_consistency.values() if v == 100)
print(f"{'Features with 100% sign consistency':<40} {consistent_features:>10}/{len(all_feature_names)} {'PASS' if consistent_features > len(all_feature_names)*0.7 else 'REVIEW':>10}")

## Step 5: Model Registry

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name=DATABASE, schema_name=ML_SCHEMA)

sample_input = session.create_dataframe(X.head(10))
version_name = "V01"

model_version = registry.log_model(
    model=final_model,
    model_name="CREDIT_RISK_XGBOOST",
    version_name=version_name,
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"],
    metrics={
        "mean_auc": float(mean_auc),
        "mean_f1": float(mean_f1),
        "n_samples": len(X),
        "default_rate": float(y.mean()),
    },
    comment="XGBoost credit risk model: PD prediction from financial ratios + covenant health + regime"
)

print(f"Model logged: CREDIT_RISK_XGBOOST/{version_name}")
print(f"Metrics: AUC={mean_auc:.4f}, F1={mean_f1:.4f}")

## Step 6: ML Pipeline Deployment

In [ ]:
from snowflake.core import Root
from snowflake.core.task import Cron
from snowflake.core.task.dagv1 import DAG, DAGTask, DAGOperation

dag = DAG(
    name="CREDIT_RISK_PIPELINE",
    schedule=Cron("0 8 1 * *", "America/New_York"),
    warehouse="SAM_DEMO_EXECUTION_WH"
)

score_task = DAGTask(
    name="SCORE_CREDIT_RISK",
    definition=f"""
        INSERT INTO {DATABASE}.{ML_SCHEMA}.FACT_CREDIT_RISK_SCORES
        SELECT
            BORROWERID,
            QUARTER_DATE,
            MODEL({DATABASE}.{ML_SCHEMA}.CREDIT_RISK_XGBOOST, '{version_name}')!predict(
                TOTALLEVERAGE, NETLEVERAGE, INTERESTCOVERAGE, FIXEDCHARGECOVERAGE,
                EBITDA_MARGIN, FCF_MARGIN, CAPEX_INTENSITY, FREECASHFLOW_MM,
                CASHONHAND_MM, REVOLVERAVAILABILITY_MM, AVG_COVENANT_HEADROOM,
                BREACH_COUNT, MIN_COVENANT_HEADROOM,
                E_SCORE, S_SCORE, G_SCORE, ESG_COMPOSITE,
                SECTOR, CREDITRATING, MARKET_REGIME
            ):output_feature_0::FLOAT AS PD_SCORE,
            CASE
                WHEN PD_SCORE > 0.5 THEN 'HIGH_RISK'
                WHEN PD_SCORE > 0.2 THEN 'ELEVATED'
                WHEN PD_SCORE > 0.1 THEN 'MODERATE'
                ELSE 'LOW_RISK'
            END AS RISK_RATING,
            NULL AS SHAP_TOP_FEATURES,
            '{version_name}' AS MODEL_VERSION,
            CURRENT_TIMESTAMP() AS SCORED_AT
        FROM TABLE({DATABASE}.{ML_SCHEMA}.CREDIT_RISK_FEATURES$V01)
        WHERE QUARTER_DATE = (SELECT MAX(QUARTER_DATE) FROM TABLE({DATABASE}.{ML_SCHEMA}.CREDIT_RISK_FEATURES$V01))
    """,
    warehouse="SAM_DEMO_EXECUTION_WH"
)

dag.add_task(score_task)

root = Root(session)
schema_ref = root.databases[DATABASE].schemas[ML_SCHEMA]
dag_op = DAGOperation(schema_ref)
dag_op.deploy(dag)
print(f"Pipeline deployed: CREDIT_RISK_PIPELINE (monthly, 1st of month at 08:00 ET)")

## Step 7: ML Observability — Model Monitor

In [ ]:
baseline_credit = X.copy()
baseline_credit["QUARTER_DATE"] = training_data.loc[X.index, "QUARTER_DATE"].values
baseline_credit["BORROWERID"] = training_data.loc[X.index, "BORROWERID"].values
baseline_credit["PD_PREDICTION"] = final_model.predict_proba(X)[:, 1]
baseline_credit["DEFAULT_FLAG"] = y.values

baseline_sf = session.create_dataframe(baseline_credit)
baseline_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.CREDIT_RISK_BASELINE")
print(f"Baseline table: {len(baseline_credit)} rows")

monitor_sql = f"""
CREATE OR REPLACE MODEL MONITOR {DATABASE}.{ML_SCHEMA}.CREDIT_RISK_MONITOR
WITH
    MODEL = {DATABASE}.{ML_SCHEMA}.CREDIT_RISK_XGBOOST VERSION = '{version_name}'
    SOURCE = {DATABASE}.{ML_SCHEMA}.FACT_CREDIT_RISK_SCORES
    WAREHOUSE = SAM_DEMO_EXECUTION_WH
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '7 days'
    TIMESTAMP_COLUMN = QUARTER_DATE
    PREDICTION_SCORE_COLUMNS = (PD_SCORE)
    ID_COLUMNS = (BORROWER_ID)
    BASELINE = {DATABASE}.{ML_SCHEMA}.CREDIT_RISK_BASELINE
"""
session.sql(monitor_sql).collect()
print("Model Monitor created: CREDIT_RISK_MONITOR")

## Score and Populate Prediction Tables

In [ ]:
y_pred_proba = final_model.predict_proba(X)[:, 1]

scores_df = pd.DataFrame({
    "BORROWER_ID": training_data.loc[X.index, "BORROWERID"].values,
    "QUARTER_DATE": training_data.loc[X.index, "QUARTER_DATE"].values,
    "PD_SCORE": y_pred_proba,
    "RISK_RATING": pd.cut(y_pred_proba, bins=[0, 0.1, 0.2, 0.5, 1.0],
                           labels=["LOW_RISK", "MODERATE", "ELEVATED", "HIGH_RISK"]).astype(str),
    "MODEL_VERSION": version_name,
})

scores_sf = session.create_dataframe(scores_df)
scores_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.FACT_CREDIT_RISK_SCORES")

n_samples, n_features = shap_values.shape
borrower_ids = np.repeat(training_data.loc[X.index, "BORROWERID"].values, n_features)
quarter_dates = np.repeat(training_data.loc[X.index, "QUARTER_DATE"].values.astype(str), n_features)
feature_names = np.tile(all_feature_names, n_samples)
shap_flat = shap_values.flatten()
feature_flat = X.values.flatten()

shap_df = pd.DataFrame({
    "BORROWER_ID": borrower_ids.astype(int),
    "QUARTER_DATE": quarter_dates,
    "FEATURE_NAME": feature_names,
    "SHAP_VALUE": shap_flat.astype(float),
    "FEATURE_VALUE": feature_flat.astype(float),
})

shap_sf = session.create_dataframe(shap_df)
shap_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.FACT_CREDIT_SHAP_EXPLANATIONS")

print(f"Scored {len(scores_df)} borrower-quarters into FACT_CREDIT_RISK_SCORES")
print(f"Wrote {len(shap_df)} SHAP explanations into FACT_CREDIT_SHAP_EXPLANATIONS")
print(f"\nRisk distribution:")
print(scores_df["RISK_RATING"].value_counts())

#### Output Quality Gate: Risk Rating Distribution and Monotonicity

The final PD scores are bucketed into 4 risk ratings (LOW_RISK, MODERATE, ELEVATED, HIGH_RISK). For these ratings to be useful for portfolio management, they must differentiate borrowers meaningfully and the actual default rate must increase monotonically from LOW to HIGH.

**What to look for:**
- **Rating distribution** — Should be spread across all buckets. If 90% of borrowers land in LOW_RISK, the ratings do not differentiate and are useless for risk-based pricing or limit setting.
- **Monotonic default rates** — The actual default rate must increase from LOW to HIGH. If ELEVATED borrowers default less often than MODERATE, the rating boundaries are miscalibrated.
- **PD score distribution** — The histogram of raw PD scores should show some separation (bimodal is ideal for binary classification). A single tight peak means the model assigns similar PD to everyone.
- **HHI concentration** — Herfindahl index of rating distribution. Near 0.25 = evenly spread across 4 buckets; near 1.0 = all in one bucket.

**Decision impact:** Non-monotonic default rates mean the PD bucket boundaries need adjustment. High concentration (HHI > 0.5) means the model's discrimination does not translate into useful risk tiers — consider different bucket boundaries or a finer grading scale.

In [ ]:
rating_order = ["LOW_RISK", "MODERATE", "ELEVATED", "HIGH_RISK"]

scores_table = session.table(f"{DATABASE}.{ML_SCHEMA}.FACT_CREDIT_RISK_SCORES")

rating_dist = (scores_table
    .group_by("RISK_RATING")
    .agg(F.count("*").alias("CNT"))
).to_pandas()

training_sf = session.create_dataframe(
    training_data[["BORROWERID", "QUARTER_DATE", "DEFAULT_FLAG"]]
        .rename(columns={"BORROWERID": "BORROWER_ID"})
)

default_by_rating = (scores_table
    .join(training_sf, ["BORROWER_ID", "QUARTER_DATE"], "left")
    .group_by("RISK_RATING")
    .agg(F.avg("DEFAULT_FLAG").alias("DEFAULT_RATE"))
).to_pandas()

pd_scores = scores_table.select("PD_SCORE").to_pandas()

rating_counts = rating_dist.set_index("RISK_RATING")["CNT"].reindex(rating_order, fill_value=0)
rating_default_rates = default_by_rating.set_index("RISK_RATING")["DEFAULT_RATE"].reindex(rating_order) * 100

rating_colors = {"LOW_RISK": "#2ecc71", "MODERATE": "#f39c12", "ELEVATED": "#e67e22", "HIGH_RISK": "#e74c3c"}

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Risk Rating Distribution", "Actual Default Rate by Rating",
    "PD Score Distribution with Rating Boundaries"])

r_colors = [rating_colors.get(r, "#999") for r in rating_counts.index]
fig.add_trace(go.Bar(x=list(rating_counts.index), y=rating_counts.values,
    marker_color=r_colors, text=[str(c) for c in rating_counts.values],
    textposition="outside", showlegend=False), row=1, col=1)
fig.update_yaxes(title_text="# Borrower-Quarters", row=1, col=1)

dr_colors = [rating_colors.get(r, "#999") for r in rating_default_rates.index]
dr_text = [f"{v:.1f}%" if not pd.isna(v) else "" for v in rating_default_rates.values]
fig.add_trace(go.Bar(x=list(rating_default_rates.index), y=rating_default_rates.values,
    marker_color=dr_colors, text=dr_text, textposition="outside", showlegend=False), row=1, col=2)
fig.update_yaxes(title_text="Default Rate (%)", row=1, col=2)

is_monotonic = all(rating_default_rates.iloc[i] <= rating_default_rates.iloc[i+1]
                   for i in range(len(rating_default_rates)-1)
                   if not pd.isna(rating_default_rates.iloc[i]) and not pd.isna(rating_default_rates.iloc[i+1]))

fig.add_trace(go.Histogram(x=pd_scores["PD_SCORE"], nbinsx=50, opacity=0.7,
    marker_color="#3498db", showlegend=False), row=1, col=3)
for boundary in [0.1, 0.2, 0.5]:
    fig.add_vline(x=boundary, line_dash="dash", line_color="red", opacity=0.5, row=1, col=3)
fig.update_xaxes(title_text="PD Score", row=1, col=3)

fig.update_layout(height=380, template="plotly_white")
fig.show()

rating_shares = rating_counts / rating_counts.sum()
hhi = (rating_shares ** 2).sum()

print(f"{'Metric':<40} {'Value':>12} {'Status':>10}")
print("-" * 65)
print(f"{'Rating HHI (concentration)':<40} {hhi:>12.3f} {'PASS' if hhi < 0.5 else 'REVIEW':>10}")
print(f"{'Monotonic default rates':<40} {'Yes' if is_monotonic else 'No':>12} {'PASS' if is_monotonic else 'FAIL':>10}")
print(f"{'# ratings with > 0 borrowers':<40} {(rating_counts > 0).sum():>12} {'PASS' if (rating_counts > 0).sum() == 4 else 'REVIEW':>10}")
print(f"{'Smallest bucket %':<40} {rating_shares.min()*100:>11.1f}% {'PASS' if rating_shares.min() > 0.05 else 'REVIEW':>10}")

## Summary

| Capability | What We Used |
|---|---|
| **Feature Store** | Entity + FeatureView (borrower-level features from financials + covenants + regime) |
| **Experiment Tracking** | `ExperimentTracking` + `SnowflakeXgboostCallback` for autologging |
| **Model Registry** | `registry.log_model()` with AUC/F1 metrics, `target_platforms=['WAREHOUSE']` |
| **SHAP Explainability** | TreeExplainer for individual and portfolio-level feature importance |
| **Model Monitor** | `CREATE MODEL MONITOR` for drift detection on financial ratio distributions |
| **ML Pipeline** | DAG API for monthly scoring cycle |

The predictive model replaces covenant-only monitoring with a forward-looking probability of default that incorporates financial health, market regime, and cross-asset signals.